# Step 4. Spectral Interpolation

We can do the spatial or spectral step first.  In this case, we choose the spectral step first because the H$_2$CS cube is narrower in velocity (`cube1vel`) and this will reduce the number of channels we need to spatially interpolate over in the next step.

We need to match resolution to the cube with the largest channel width:

In [2]:
from spectral_cube import SpectralCube
from astropy import units as u
import numpy as np
import matplotlib.pyplot as plt
from radio_beam import Beam
from astropy.io import fits


In [3]:
# with fits.open("./smooth/340test.450.smooth.fits",mode="update") as hdu:
#     crafts_beam = Beam(major=4*u.arcmin, minor=4*u.arcmin, pa=0*u.deg)
#     hdu[0].header.update(crafts_beam.to_header_keywords())
#     hdu.flush()

In [5]:
cube1=SpectralCube.read('./cut/catalog-with_rms/HI4PI_-4.7_100_150_Original.fits')
cube1.with_spectral_unit(u.km/u.s, velocity_convention='radio')

SpectralCube with shape=(39, 28, 47) and unit=K:
 n_x:     47  type_x: RA---CAR  unit_x: deg    range:   353.000000 deg:  356.833333 deg
 n_y:     28  type_y: DEC--CAR  unit_y: deg    range:    -6.916667 deg:   -4.666667 deg
 n_s:     39  type_s: VRAD      unit_s: km / s  range:      100.582 km / s:     149.534 km / s

In [6]:
cube1vel = cube1.with_spectral_unit(
    u.m / u.s, velocity_convention="radio"
)
cube1vel

SpectralCube with shape=(39, 28, 47) and unit=K:
 n_x:     47  type_x: RA---CAR  unit_x: deg    range:   353.000000 deg:  356.833333 deg
 n_y:     28  type_y: DEC--CAR  unit_y: deg    range:    -6.916667 deg:   -4.666667 deg
 n_s:     39  type_s: VRAD      unit_s: m / s  range:   100581.725 m / s:  149533.894 m / s

In [7]:
cube2=SpectralCube.read('./cut/catalog-with_rms/CRAFTS_-4.7_100_150_Original.fits')
cube2.with_spectral_unit(u.km/u.s, velocity_convention='radio')

SpectralCube with shape=(250, 89, 153) and unit=K:
 n_x:    153  type_x: RA---CAR  unit_x: deg    range:   352.987500 deg:  356.787500 deg
 n_y:     89  type_y: DEC--CAR  unit_y: deg    range:    -6.887500 deg:   -4.687500 deg
 n_s:    250  type_s: VOPT      unit_s: km / s  range:       99.966 km / s:     150.086 km / s

In [8]:
cube2vel = cube2.with_spectral_unit(
    u.m / u.s, velocity_convention="radio"
)
cube2vel

SpectralCube with shape=(250, 89, 153) and unit=K:
 n_x:    153  type_x: RA---CAR  unit_x: deg    range:   352.987500 deg:  356.787500 deg
 n_y:     89  type_y: DEC--CAR  unit_y: deg    range:    -6.887500 deg:   -4.687500 deg
 n_s:    250  type_s: VOPT      unit_s: m / s  range:    99965.887 m / s:  150085.502 m / s

In [9]:
velocity_res_1 = np.diff(cube1vel.spectral_axis)[0]
velocity_res_2 = np.diff(cube2vel.spectral_axis)[0]
velocity_res_1, velocity_res_2

(<Quantity 1288.21496912 m / s>, <Quantity -201.28359473 m / s>)

In [10]:
cube2vel_cutout = cube2vel.spectral_slab(
    cube1vel.spectral_axis.min(), cube1vel.spectral_axis.max()
)
cube1vel, cube2vel_cutout

(SpectralCube with shape=(39, 28, 47) and unit=K:
  n_x:     47  type_x: RA---CAR  unit_x: deg    range:   353.000000 deg:  356.833333 deg
  n_y:     28  type_y: DEC--CAR  unit_y: deg    range:    -6.916667 deg:   -4.666667 deg
  n_s:     39  type_s: VRAD      unit_s: m / s  range:   100581.725 m / s:  149533.894 m / s,
 SpectralCube with shape=(244, 89, 153) and unit=K:
  n_x:    153  type_x: RA---CAR  unit_x: deg    range:   352.987500 deg:  356.787500 deg
  n_y:     89  type_y: DEC--CAR  unit_y: deg    range:    -6.887500 deg:   -4.687500 deg
  n_s:    244  type_s: VOPT      unit_s: m / s  range:   100569.738 m / s:  149481.651 m / s)

In [11]:
cube2vel_cutout = cube2vel.spectral_slab(
    cube1vel.spectral_axis.min() - velocity_res_2, cube1vel.spectral_axis.max()
)
cube1vel, cube2vel_cutout

(SpectralCube with shape=(39, 28, 47) and unit=K:
  n_x:     47  type_x: RA---CAR  unit_x: deg    range:   353.000000 deg:  356.833333 deg
  n_y:     28  type_y: DEC--CAR  unit_y: deg    range:    -6.916667 deg:   -4.666667 deg
  n_s:     39  type_s: VRAD      unit_s: m / s  range:   100581.725 m / s:  149533.894 m / s,
 SpectralCube with shape=(243, 89, 153) and unit=K:
  n_x:    153  type_x: RA---CAR  unit_x: deg    range:   352.987500 deg:  356.787500 deg
  n_y:     89  type_y: DEC--CAR  unit_y: deg    range:    -6.887500 deg:   -4.687500 deg
  n_s:    243  type_s: VOPT      unit_s: m / s  range:   100771.021 m / s:  149481.651 m / s)

In [12]:
fwhm_gaussian = (velocity_res_1**2 - velocity_res_2**2) ** 0.5
fwhm_gaussian

<Quantity 1272.39251851 m / s>

In [13]:
from astropy.convolution import Gaussian1DKernel

fwhm_to_sigma = np.sqrt(8 * np.log(2))
# we want the kernel in pixel units, so we force to km/s and take the value
spectral_smoothing_kernel = Gaussian1DKernel(
    stddev=fwhm_gaussian.to(u.km / u.s).value / fwhm_to_sigma
)

In [14]:
cube2vel_smooth = cube2vel_cutout.spectral_smooth(spectral_smoothing_kernel)

In [15]:
cube2vel_spectralresample = cube2vel_smooth.spectral_interpolate(
    cube1vel.spectral_axis, suppress_smooth_warning=True
)
cube2vel_spectralresample

Spectral Interpolate:   0%|          | 0/13617 [00:00<?, ?it/s]

SpectralCube with shape=(39, 89, 153) and unit=K:
 n_x:    153  type_x: RA---CAR  unit_x: deg    range:   352.987500 deg:  356.787500 deg
 n_y:     89  type_y: DEC--CAR  unit_y: deg    range:    -6.887500 deg:   -4.687500 deg
 n_s:     39  type_s: VOPT      unit_s: m / s  range:   100581.725 m / s:  149533.894 m / s

In [16]:
cube2vel_spectralresample.header

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -32 / array data type                                
NAXIS   =                    3                                                  
NAXIS1  =                  153                                                  
NAXIS2  =                   89                                                  
NAXIS3  =                   39                                                  
DATE    = '2024-08-07'         / Creation UTC (CCCC-MM-DD) date of FITS header  
COMMENT FITS (Flexible Image Transport System) format is defined in 'Astronomy  
COMMENT and Astrophysics', volume 376, page 359; bibcode 2001A&A...376..359H    
COMMENT Original Unit=m / s                                                     
COMMENT Original Type=VOPT                                                      
EPOCH   =              2000.00 /                                                
BPA     =              0.000

In [17]:
cube2vel_spectralresample.write("./cut/catalog-with_rms/CRAFTS_-4.7_100_150_Original_smooth-velocity.fits")

In [18]:
from astropy.io import fits

with fits.open('./cut/catalog-with_rms/CRAFTS_-4.7_100_150_Original_smooth-velocity.fits',
               mode='update') as hdul:

    header = hdul[0].header

    # 正确写法：value + unit
    header['BPA']  = (0.0, 'deg')
    header['BMAJ'] = (0.066666667, 'deg')
    header['BMIN'] = (0.066666667, 'deg')

    hdul.flush()

# 5

In [6]:
from radio_beam import Beam
from astropy.io import fits


In [7]:

from radio_beam import Beam
import astropy.units as u
from spectral_cube import SpectralCube

cube = SpectralCube.read("./cut/new_baseline/CRAFTS_-4.7_-450_-350_baseline_smooth-velocity.fits")

beam = Beam(240*u.arcsec, 240*u.arcsec, 0*u.deg)

cube = cube.with_beam(beam)

print(cube.beam)

Beam: BMAJ=240.0 arcsec BMIN=240.0 arcsec BPA=0.0 deg


In [8]:
my_beam_ellip = Beam(major=974.16*u.arcsec, minor=974.16*u.arcsec, pa=0*u.deg)
my_beam_ellip

Beam: BMAJ=974.16 arcsec BMIN=974.16 arcsec BPA=0.0 deg

In [9]:
# for v<0.6, we convert to Kelvin to ensure the units are preserved:
# cube2vel_spatialspectralsmooth = cube2vel_spectralresample.to(u.K).convolve_to(common_beam)
# in more recent versions, the unit conversion is handled appropriately,
# so unit conversion isn't needed

cube2vel_spatialspectralsmooth = cube.convolve_to(my_beam_ellip)
cube2vel_spatialspectralsmooth

SpectralCube with shape=(78, 89, 153) and unit=K:
 n_x:    153  type_x: RA---CAR  unit_x: deg    range:   352.987500 deg:  356.787500 deg
 n_y:     89  type_y: DEC--CAR  unit_y: deg    range:    -6.887500 deg:   -4.687500 deg
 n_s:     78  type_s: VRAD-W2F  unit_s: km / s  range:     -449.486 km / s:    -350.326 km / s

In [10]:
cube2vel_spatialspectralsmooth.write('./cut/new_baseline/CRAFTS_-4.7_-450_-350_baseline_smooth-velocity-beam.fits',overwrite="True")

In [46]:
import astropy
import spectral_cube
import radio_beam

print("astropy =", astropy.__version__)
print("spectral_cube =", spectral_cube.__version__)
print("radio_beam =", radio_beam.__version__)

astropy = 7.2.0
spectral_cube = 0.6.5
radio_beam = 0.3.7


In [47]:
pip install -U radio-beam spectral-cube

Looking in indexes: https://mirrors.cloud.tencent.com/pypi/simple
     ---------------------------------------- 0.0/73.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/73.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/73.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/73.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/73.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/73.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/73.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/73.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/73.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/73.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/73.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/73.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/73.3 kB ? eta -:--:-

In [5]:
import numpy as np
import astropy.units as u
from astropy.convolution import Gaussian1DKernel
from spectral_cube import SpectralCube

# =========================
# 1. 读取 cube
# =========================
cube_crafts = SpectralCube.read(
    "./cut/new_baseline/CRAFTS_-4.7_-450_-350_baseline.fits"
).with_spectral_unit(u.km/u.s, velocity_convention="radio")

cube_hi4pi = SpectralCube.read(
    "./cut/new_baseline/HI4PI_-4.7_-450_-350_baseline.fits"
).with_spectral_unit(u.km/u.s, velocity_convention="radio")

# =========================
# 2. 获取速度通道间隔
# =========================
dv_crafts = np.abs(np.nanmedian(np.diff(cube_crafts.spectral_axis.to_value(u.km/u.s))))
dv_hi4pi = np.abs(np.nanmedian(np.diff(cube_hi4pi.spectral_axis.to_value(u.km/u.s))))

print("CRAFTS dv =", dv_crafts, "km/s")
print("HI4PI dv  =", dv_hi4pi, "km/s")

# =========================
# 3. 计算需要卷积的高斯核
# =========================
fwhm_crafts = dv_crafts
fwhm_hi4pi = dv_hi4pi

fwhm_kernel = np.sqrt(fwhm_hi4pi**2 - fwhm_crafts**2)
sigma_kernel_kms = fwhm_kernel / 2.354820045
sigma_kernel_pix = sigma_kernel_kms / dv_crafts

print("Kernel FWHM =", fwhm_kernel, "km/s")
print("Kernel sigma =", sigma_kernel_kms, "km/s")
print("Kernel sigma =", sigma_kernel_pix, "channels")

# =========================
# 4. 建立高斯核并做 spectral smoothing
# =========================
kernel = Gaussian1DKernel(stddev=sigma_kernel_pix)

cube_crafts_smooth = cube_crafts.spectral_smooth(kernel)

# =========================
# 5. 插值到 HI4PI 速度轴
# =========================
cube_crafts_smooth_interp = cube_crafts_smooth.spectral_interpolate(
    cube_hi4pi.spectral_axis
)

print(cube_crafts_smooth_interp)

# =========================
# 6. 保存结果
# =========================
output_file = "./cut/new_baseline/CRAFTS_-4.7_-450_-350_baseline_smooth-velocity.fits"

cube_crafts_smooth_interp.write(
    output_file,
    overwrite=True
)

print("Saved to:", output_file)

CRAFTS dv = 0.20182169418211515 km/s
HI4PI dv  = 1.2882149691241693 km/s
Kernel FWHM = 1.2723072783070313 km/s
Kernel sigma = 0.5402991540727399 km/s
Kernel sigma = 2.6771113792415067 channels


Spectral Interpolate:   0%|          | 0/13617 [00:00<?, ?it/s]

SpectralCube with shape=(78, 89, 153) and unit=K:
 n_x:    153  type_x: RA---CAR  unit_x: deg    range:   352.987500 deg:  356.787500 deg
 n_y:     89  type_y: DEC--CAR  unit_y: deg    range:    -6.887500 deg:   -4.687500 deg
 n_s:     78  type_s: VRAD-W2F  unit_s: km / s  range:     -449.486 km / s:    -350.326 km / s
Saved to: ./cut/new_baseline/CRAFTS_-4.7_-450_-350_baseline_smooth-velocity.fits


In [3]:
import numpy as np
import astropy.units as u
from astropy.io import fits
from spectral_cube import SpectralCube
from reproject import reproject_interp
from tqdm import tqdm

# ============================================================
# 1. 读取已经平滑后的 CRAFTS cube
#    这里建议使用已经：
#    1) velocity smooth 到 HI4PI
#    2) beam convolve 到 974.16 arcsec
#    的 CRAFTS cube
# ============================================================

crafts_file = "./cut/new_baseline/CRAFTS_-4.7_-350_-150_baseline_smooth-velocity-beam.fits"

cube_crafts = SpectralCube.read(crafts_file).with_spectral_unit(
    u.km/u.s,
    velocity_convention="radio"
)

print("CRAFTS cube:")
print(cube_crafts)
print("CRAFTS shape:", cube_crafts.shape)
print("CRAFTS pixel scale should be ~0.025 deg")


# ============================================================
# 2. 读取 HI4PI cube，用它的空间 WCS 作为目标网格
# ============================================================

hi4pi_file = "./cut/new_baseline/HI4PI_-4.7_-350_-150_Original.fits"

cube_hi4pi = SpectralCube.read(hi4pi_file).with_spectral_unit(
    u.km/u.s,
    velocity_convention="radio"
)

print("HI4PI cube:")
print(cube_hi4pi)
print("HI4PI shape:", cube_hi4pi.shape)
print("HI4PI pixel scale should be ~0.083333 deg")


# ============================================================
# 3. 提取目标空间 header
# ============================================================

# HI4PI 的空间大小
ny_new = cube_hi4pi.shape[1]
nx_new = cube_hi4pi.shape[2]

# HI4PI 的 celestial WCS header
target_header_2d = cube_hi4pi.wcs.celestial.to_header()
target_header_2d["NAXIS"] = 2
target_header_2d["NAXIS1"] = nx_new
target_header_2d["NAXIS2"] = ny_new

print("Target spatial shape:", ny_new, nx_new)
print("Target CDELT1:", target_header_2d["CDELT1"])
print("Target CDELT2:", target_header_2d["CDELT2"])


# ============================================================
# 4. 逐个 velocity channel 做空间 reproject
# ============================================================

data_crafts = cube_crafts.unmasked_data[:].value
nv = data_crafts.shape[0]

data_regrid = np.full(
    (nv, ny_new, nx_new),
    np.nan,
    dtype=np.float32
)

for k in tqdm(range(nv), desc="Reprojecting spatial slices"):

    image_2d = data_crafts[k, :, :]

    # 每个 channel 是一张 2D 图，使用 CRAFTS 的 celestial WCS
    input_hdu = fits.PrimaryHDU(
        data=image_2d,
        header=cube_crafts.wcs.celestial.to_header()
    )

    reprojected_image, footprint = reproject_interp(
        input_hdu,
        target_header_2d,
        order="bilinear"
    )

    data_regrid[k, :, :] = reprojected_image.astype(np.float32)


# ============================================================
# 5. 构建新的 3D header
# ============================================================

header_out = cube_crafts.header.copy()

# 替换空间 WCS 为 HI4PI 的空间 WCS
for key in [
    "CTYPE1", "CTYPE2",
    "CUNIT1", "CUNIT2",
    "CRVAL1", "CRVAL2",
    "CRPIX1", "CRPIX2",
    "CDELT1", "CDELT2",
    "PC1_1", "PC1_2", "PC2_1", "PC2_2",
    "CD1_1", "CD1_2", "CD2_1", "CD2_2"
]:
    if key in target_header_2d:
        header_out[key] = target_header_2d[key]
    elif key in header_out and key.startswith("CD"):
        # 如果原 header 有 CD 矩阵，但 target 没有，可以删除，避免冲突
        del header_out[key]

# 更新尺寸
header_out["NAXIS"] = 3
header_out["NAXIS1"] = nx_new
header_out["NAXIS2"] = ny_new
header_out["NAXIS3"] = nv

# 保留 beam 信息
# 如果你的 CRAFTS 已经卷积到 HI4PI beam，应该是 974.16 arcsec = 0.2706 deg
header_out["BMAJ"] = 974.16 / 3600.0
header_out["BMIN"] = 974.16 / 3600.0
header_out["BPA"] = 0.0

# 记录历史
header_out["HISTORY"] = "Spatially reprojected from CRAFTS 0.025 deg pixels to HI4PI 0.083333 deg pixels."


# ============================================================
# 6. 保存
# ============================================================

output_file = "./cut/new_baseline/CRAFTS_-4.7_-350_-150_baseline_smooth-velocity-beam-HI4PIpixel.fits"

fits.writeto(
    output_file,
    data_regrid,
    header_out,
    overwrite=True
)

print("Saved regridded cube to:")
print(output_file)

CRAFTS cube:
SpectralCube with shape=(156, 89, 153) and unit=K:
 n_x:    153  type_x: RA---CAR  unit_x: deg    range:   352.987500 deg:  356.787500 deg
 n_y:     89  type_y: DEC--CAR  unit_y: deg    range:    -6.887500 deg:   -4.687500 deg
 n_s:    156  type_s: VRAD      unit_s: km / s  range:     -350.294 km / s:    -150.620 km / s
CRAFTS shape: (156, 89, 153)
CRAFTS pixel scale should be ~0.025 deg
HI4PI cube:
SpectralCube with shape=(156, 28, 47) and unit=K:
 n_x:     47  type_x: RA---CAR  unit_x: deg    range:   353.000000 deg:  356.833333 deg
 n_y:     28  type_y: DEC--CAR  unit_y: deg    range:    -6.916667 deg:   -4.666667 deg
 n_s:    156  type_s: VRAD      unit_s: km / s  range:     -350.294 km / s:    -150.620 km / s
HI4PI shape: (156, 28, 47)
HI4PI pixel scale should be ~0.083333 deg
Target spatial shape: 28 47
Target CDELT1: -0.0833333333
Target CDELT2: 0.0833333333


Reprojecting spatial slices: 100%|███████████████████████████████████████████████████| 156/156 [00:04<00:00, 33.31it/s]

Saved regridded cube to:
./cut/new_baseline/CRAFTS_-4.7_-350_-150_baseline_smooth-velocity-beam-HI4PIpixel.fits


In [4]:
cube_check = SpectralCube.read(
    "./cut/new_baseline/CRAFTS_-4.7_-350_-150_baseline_smooth-velocity-beam-HI4PIpixel.fits"
)

print(cube_check)
print(cube_check.header["CDELT1"], cube_check.header["CDELT2"])
print(cube_check.header["BMAJ"], cube_check.header["BMIN"])

SpectralCube with shape=(156, 28, 47) and unit=K:
 n_x:     47  type_x: RA---CAR  unit_x: deg    range:   353.000000 deg:  356.833333 deg
 n_y:     28  type_y: DEC--CAR  unit_y: deg    range:    -6.916667 deg:   -4.666667 deg
 n_s:    156  type_s: VRAD      unit_s: km / s  range:     -350.294 km / s:    -150.620 km / s
-0.0833333333 0.0833333333
0.2706 0.2706
